# Feature Engineering

Builds the per-horizon feature tables used by all four models (XGBoost, MLP, LSTM, GNN), using the train/test parquet files produced by the `1_raw_data_prep.ipynb` notebook.

Encoders (`train_type`, `elektrifizierung`) and the station average-delay lookup are fit on the training split only, then applied to both splits, and saved so every model uses identical encodings.

Each horizon's train and test tables are saved as their own parquet file, with non-horizon-suffixed column names. Every candidate feature group is included in these files.

In [1]:
import polars as pl
import numpy as np
from sklearn.preprocessing import LabelEncoder

## Configuration

`N_FUTURE` is the number of future stops each snapshot has a target for. `N_PAST_HISTORY` is how many past stops' delay/timing are available as features. The candidate group lists match the feature-group taxonomy used throughout the ablation and feature-selection. Each model later excludes whichever of these groups it has confirmed don't help, at any horizons that exclusion applies.

`CYCLICAL_FREQUENCIES` represents hour-of-day and day-of-year at multiple harmonics (1, 2, 4) rather than a single sine/cosine pair.

In [2]:
DATA_DIR = "/kaggle/input/datasets/ranjithpanicker/railway-data"
TRAIN_PATH = f"{DATA_DIR}/train_3city_2025-11_to_2026-04_final.parquet"
TEST_PATH = f"{DATA_DIR}/test_3city_2026-05_to_2026-07_final.parquet"
OUTPUT_DIR = "/kaggle/working/features"

N_FUTURE = 10
N_PAST_HISTORY = 3

WEATHER_VARS = ["temperature_2m", "precipitation", "wind_speed_10m"]
INFRA_NUMERIC_VARS = ["gleisanzahl", "geschwindigkeit"]
TIMETABLE_VARS = ["station_headway_min", "station_dwell_min", "station_freq_per_day"]
CONGESTION_AHEAD_VARS = ["station_congestion_n", "station_congestion_avgdelay"]
INFRA_CATEGORICAL_VAR = "elektrifizierung"
CYCLICAL_FREQUENCIES = [1, 2, 4]

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load the raw train and test tables

One row per snapshot. Each snapshot carries the target train's own recent history (`last_known_delay`, `past_delay_1..3`, `past_minutes_ago_1..3`), station-level context at the snapshot station, and, for each of the ten future stops, that stop's delay, elapsed time ahead, station name, and weather/infrastructure/timetable/congestion values under a `future_<var>_<horizon>` naming convention.

In [3]:
train_df = pl.read_parquet(TRAIN_PATH)
test_df = pl.read_parquet(TEST_PATH)
print(f"Train: {train_df.height:,} rows. Test: {test_df.height:,} rows.")

Train: 2,460,063 rows. Test: 248,055 rows.


## Fit encoders on the training split only

`train_type` and `elektrifizierung` are both categorical. Both encoders are fit on training values plus an explicit `UNKNOWN` class, so a category seen only at test time (or not at all) falls back to `UNKNOWN` rather than the encoder raising on an unseen label.

In [4]:
def fit_label_encoder(values):
    le = LabelEncoder()
    le.fit(list(values) + ["UNKNOWN"])
    return le

def encode_categorical(values, label_encoder):
    known = set(label_encoder.classes_)
    safe = [v if v in known else "UNKNOWN" for v in values]
    return label_encoder.transform(safe)

train_type_encoder = fit_label_encoder(train_df["train_type"].fill_null("UNKNOWN").unique().to_list())

elektrifizierung_values = set()
for i in range(1, N_FUTURE + 1):
    col = f"future_{INFRA_CATEGORICAL_VAR}_{i}"
    if col in train_df.columns:
        elektrifizierung_values.update(train_df[col].drop_nulls().unique().to_list())
elektrifizierung_encoder = fit_label_encoder(elektrifizierung_values)

print(f"train_type classes: {list(train_type_encoder.classes_)}")
print(f"elektrifizierung classes: {list(elektrifizierung_encoder.classes_)}")

train_type classes: [np.str_('ARV'), np.str_('BRB'), np.str_('D'), np.str_('EC'), np.str_('EN'), np.str_('ES'), np.str_('FEX'), np.str_('FLX'), np.str_('IC'), np.str_('ICE'), np.str_('ME'), np.str_('NBE'), np.str_('NEB'), np.str_('NJ'), np.str_('OE'), np.str_('RB'), np.str_('RE'), np.str_('RJ'), np.str_('RSM'), np.str_('RSU'), np.str_('S'), np.str_('SMD'), np.str_('UEX'), np.str_('UNKNOWN'), np.str_('WB')]
elektrifizierung classes: [np.str_('Oberleitung'), np.str_('Stromschiene'), np.str_('UNKNOWN'), np.str_('nicht elektrifiziert')]


## Station average-delay lookup

For each station, the average delay observed in training data is calculated. Built once from training data only, then applied identically to both train/test splits as a plain dictionary lookup, with the global average as the fallback for a station never seen in training.

In [5]:
station_delay_pairs = []
for i in range(1, N_FUTURE + 1):
    station_col, delay_col = f"future_station_{i}", f"future_delay_{i}"
    if station_col in train_df.columns and delay_col in train_df.columns:
        pair = train_df.select([pl.col(station_col).alias("station"), pl.col(delay_col).alias("delay")]).drop_nulls()
        station_delay_pairs.append(pair)

pooled_station_delays = pl.concat(station_delay_pairs, how="vertical_relaxed")
station_lookup_df = pooled_station_delays.group_by("station").agg(pl.col("delay").mean().alias("avg_delay"))
station_lookup = dict(zip(station_lookup_df["station"].to_list(), station_lookup_df["avg_delay"].to_list()))
global_avg_delay = float(pooled_station_delays["delay"].mean())

print(f"{len(station_lookup)} stations in lookup, global average delay = {global_avg_delay:.2f} min")

183 stations in lookup, global average delay = 3.23 min


## Cyclical time features

Hour-of-day and day-of-year are each encoded as sine/cosine pairs at three harmonics, capturing daily patterns with more than one peak and seasonal patterns across the six-month training window. Day-of-week is encoded both as a single sine/cosine pair and as one-hot columns.


In [6]:
def add_cyclical_time_features(df):
    exprs = []
    for freq in CYCLICAL_FREQUENCIES:
        exprs.append((2 * np.pi * freq * pl.col("snapshot_hour") / 24).sin().alias(f"hour_sin_{freq}"))
        exprs.append((2 * np.pi * freq * pl.col("snapshot_hour") / 24).cos().alias(f"hour_cos_{freq}"))
    for freq in CYCLICAL_FREQUENCIES:
        exprs.append((2 * np.pi * freq * pl.col("snapshot_time").dt.ordinal_day() / 365.25).sin().alias(f"doy_sin_{freq}"))
        exprs.append((2 * np.pi * freq * pl.col("snapshot_time").dt.ordinal_day() / 365.25).cos().alias(f"doy_cos_{freq}"))
    exprs.append((2 * np.pi * pl.col("snapshot_dow") / 7).sin().alias("dow_sin"))
    exprs.append((2 * np.pi * pl.col("snapshot_dow") / 7).cos().alias("dow_cos"))
    for d in range(7):
        exprs.append((pl.col("snapshot_dow") == d).cast(pl.Int8).alias(f"dow_onehot_{d}"))
    return df.with_columns(exprs)

train_df = add_cyclical_time_features(train_df)
test_df = add_cyclical_time_features(test_df)

## Per-horizon feature table

For a given horizon, pulls together the target train's own history, the cyclical time features, station-level congestion at the snapshot station, the encoded categorical features, the station average-delay lookup, and every weather/infrastructure/timetable/congestion-ahead column for that specific future stop, all under (non-horizon-suffixed) names. 

The target train's own history, its station-level congestion counts, and `past_minutes_ago` are null-filled with zero.

`avg_delay_others_at_current_station` in particular is genuinely undefined whenever a train is the only one at its station at that moment, so nulls there are expected, not a data error, and zero encodes the reasonable default of "no evidence of other trains running late nearby." 

`minutes_ahead` and the weather/infrastructure/timetable/congestion-ahead columns use median imputation, since a genuinely missing weather reading has no equivalent zero value.

This is applied to both splits at every horizon.

In [7]:
def build_horizon_features(df, horizon_i):
    target_col = f"future_delay_{horizon_i}"
    minutes_ahead_col = f"future_minutes_ahead_{horizon_i}"
    station_col = f"future_station_{horizon_i}"
    elektrifizierung_col = f"future_{INFRA_CATEGORICAL_VAR}_{horizon_i}"

    valid = df.filter(pl.col(target_col).is_not_null())

    out = valid.select(
        ["snapshot_time", "current_station", "last_known_delay"]
        + [f"past_delay_{i}" for i in range(1, N_PAST_HISTORY + 1)]
        + [f"past_minutes_ago_{i}" for i in range(1, N_PAST_HISTORY + 1)]
        + [f"hour_sin_{f}" for f in CYCLICAL_FREQUENCIES] + [f"hour_cos_{f}" for f in CYCLICAL_FREQUENCIES]
        + [f"doy_sin_{f}" for f in CYCLICAL_FREQUENCIES] + [f"doy_cos_{f}" for f in CYCLICAL_FREQUENCIES]
        + ["dow_sin", "dow_cos"] + [f"dow_onehot_{d}" for d in range(7)]
        + ["n_trains_at_current_station", "avg_delay_others_at_current_station"]
    )

    passthrough_numeric_cols = (
        ["last_known_delay"] + [f"past_delay_{i}" for i in range(1, N_PAST_HISTORY + 1)]
        + [f"past_minutes_ago_{i}" for i in range(1, N_PAST_HISTORY + 1)]
        + ["n_trains_at_current_station", "avg_delay_others_at_current_station"]
    )
    out = out.with_columns([pl.col(c).fill_null(0) for c in passthrough_numeric_cols])

    train_type_enc = encode_categorical(valid["train_type"].fill_null("UNKNOWN").to_list(), train_type_encoder)
    out = out.with_columns(pl.Series("train_type_enc", train_type_enc))

    minutes_ahead = valid[minutes_ahead_col].to_numpy().astype(float)
    minutes_ahead = np.where(np.isnan(minutes_ahead), np.nanmedian(minutes_ahead), minutes_ahead)
    out = out.with_columns(pl.Series("minutes_ahead", minutes_ahead))

    stations = valid[station_col].to_list()
    station_avg_delay = [station_lookup.get(s, global_avg_delay) if s is not None else global_avg_delay for s in stations]
    out = out.with_columns(pl.Series("station_avg_delay", station_avg_delay))

    for group_vars in (WEATHER_VARS, INFRA_NUMERIC_VARS, TIMETABLE_VARS, CONGESTION_AHEAD_VARS):
        for var in group_vars:
            raw_col = f"future_{var}_{horizon_i}"
            values = valid[raw_col].to_numpy().astype(float)
            values = np.where(np.isnan(values), np.nanmedian(values), values)
            out = out.with_columns(pl.Series(var, values))

    elektrifizierung_enc = encode_categorical(valid[elektrifizierung_col].fill_null("UNKNOWN").to_list(), elektrifizierung_encoder)
    out = out.with_columns(pl.Series("elektrifizierung_enc", elektrifizierung_enc))

    out = out.with_columns(valid[target_col].alias("target"))
    return out

## Build and save every horizon's feature table

Ten horizons, two splits each. Each file is self-contained: a model training notebook loads the horizons directly, with every candidate feature group already present, and applies its own confirmed feature exclusions at load time.

In [8]:
print("Building per-horizon feature tables...")
for horizon_i in range(1, N_FUTURE + 1):
    train_features = build_horizon_features(train_df, horizon_i)
    test_features = build_horizon_features(test_df, horizon_i)
    train_features.write_parquet(f"{OUTPUT_DIR}/train_h{horizon_i}.parquet")
    test_features.write_parquet(f"{OUTPUT_DIR}/test_h{horizon_i}.parquet")
    print(f"  horizon {horizon_i}/{N_FUTURE}: train={train_features.height:,} rows, test={test_features.height:,} rows")
print("Feature tables written.")

Building per-horizon feature tables...
  horizon 1/10: train=2,357,869 rows, test=237,612 rows
  horizon 2/10: train=1,872,437 rows, test=186,131 rows
  horizon 3/10: train=1,468,050 rows, test=145,826 rows
  horizon 4/10: train=1,183,898 rows, test=115,898 rows
  horizon 5/10: train=1,047,320 rows, test=103,644 rows
  horizon 6/10: train=952,848 rows, test=94,056 rows
  horizon 7/10: train=867,497 rows, test=85,670 rows
  horizon 8/10: train=790,832 rows, test=78,072 rows
  horizon 9/10: train=726,075 rows, test=71,496 rows
  horizon 10/10: train=668,301 rows, test=65,458 rows
Feature tables written.


## Save the encoders and lookups

Saved once here so model training notebook loads identical encodings rather than re-fitting, which would risk two models silently using different integer codes for the same category.

In [9]:
import json

with open(f"{OUTPUT_DIR}/encoders_and_lookup.json", "w") as f:
    json.dump({
        "train_type_classes": list(train_type_encoder.classes_),
        "elektrifizierung_classes": list(elektrifizierung_encoder.classes_),
        "station_lookup": station_lookup,
        "global_avg_delay": global_avg_delay,
    }, f)

print("Encoders and lookup saved.")
print("Feature engineering complete.")

Encoders and lookup saved.
Feature engineering complete.
